In [ ]:
import pandas as pd
import numpy as np
from datetime import datetime, timedelta

def generate_sales_data(filename="Shopify_Sales.csv"):
    # 1. Set timeframe: 3 years ending yesterday
    end_date = datetime.now() - timedelta(days=1)
    start_date = end_date - timedelta(days=3*365)
    date_range = pd.date_range(start=start_date, end=end_date, freq='D')

    # 2. Base parameters
    base_sales = 15000  # Median starting point
    annual_growth_rate = 0.15  # 15% growth per year

    sales_list = []

    for i, date in enumerate(date_range):
        # Year factor (Growth)
        year_index = i / 365
        growth_factor = 1 + (year_index * annual_growth_rate)

        # Seasonal factor (Sinosoidal: peaks in summer/winter, lower in spring/autumn)
        # Shifted to peak roughly in December (day 355 approx)
        seasonal_factor = 1 + 0.15 * np.sin(2 * np.pi * (date.dayofyear + 10) / 365)

        # Weekly factor (Higher sales on Friday-Sunday)
        # Monday=0, Sunday=6. We add a boost for weekend days.
        weekday_boost = 1.1 if date.weekday() >= 4 else 1.0

        # Random daily noise (Fluctuation)
        noise = np.random.uniform(0.85, 1.15)

        # Calculate daily sales
        daily_sales = base_sales * growth_factor * seasonal_factor * weekday_boost * noise

        # Clip to your requested range (12k to 25k) with some flexibility for growth
        daily_sales = max(12000, min(daily_sales, 25000 + (year_index * 5000)))

        sales_list.append(round(daily_sales, 2))

    # 3. Create DataFrame in Shopify-like format
    df = pd.DataFrame({
        'day': date_range.strftime('%Y-%m-%d'),
        'net_sales': sales_list
    })

    # Save to CSV
    df.to_csv(filename, index=False)
    print(f"✅ Successfully generated {len(df)} days of data in '{filename}'")

# Run the generator
generate_sales_data()